# Midterm: Open Duck Walking with Reference Motion and MPC
**NTUT MuJoCo Course — after Lecture 7 · 100 points**

Integrate Lectures 5–6 reference generation, Lecture 7 QP-based MPC, task-space IK,
and MuJoCo physics. Compare the same reference/position-servo baseline with and without
MPC balance feedback. The teacher solution must actually walk; kinematic playback is
not a substitute for simulation.

| Part | Points |
|---|---:|
| A: reference, named mapping, conventions and validation | 20 |
| B: LIPM MPC QP and first-action feedback | 30 |
| C: MuJoCo integration and walking experiment | 25 |
| D: comparison, plots and explanations | 20 |
| Reproducible submission | 5 |

Use bounded `for` loops and preserve TEST CELLs. Teacher files are private and are not
runtime dependencies. Disturbance recovery is an optional extension, not a basic pass condition.

In [ ]:
%matplotlib inline
import sys,os,json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.spatial import ConvexHull
for root in (Path.cwd(),*Path.cwd().parents):
    if (root/'midterm'/'robot.py').exists():
        COURSE_ROOT=root;sys.path[:0]=[str(root/'midterm'),str(root/'lecture7')];break
else: raise FileNotFoundError('Start inside the repository')
from robot import Duck
from reference import generate_reference,export_reference,interpolate_episode
from experiment import run_experiment,save_result,plot_results,render_video
from qp import solve_qp
OUTPUT=COURSE_ROOT/'midterm'/'outputs';OUTPUT.mkdir(exist_ok=True)
print('Physical simulation: 500 Hz; IK targets: 100 Hz; MPC: 50 Hz; preview: 1 second')

## Part A: Reference data are not actuator commands (20 points)

The supplied finite Placo plan starts and ends in double support. The reference supplies
CoM position/velocity/acceleration, foot poses, support labels, and ZMP. A separate
MuJoCo-model IK solve selects a valid knee branch; the original Lecture 5 generator's
joint-limit violations must not be hidden by changing limits.

The URDF has 16 movable joints and the MuJoCo model has 14 actuators. Match names;
ignore only `left_antenna` and `right_antenna`. Reject missing, unknown or duplicate names.
Course JSON quaternions are **xyzw**, MuJoCo free-joint quaternions **wxyz**. The supplied
adapter uses SLERP for orientations, named field offsets, and clamps finite recordings.
Foot coordinates in the course JSON are body-frame; task-sidecar feet are world-frame.
The existing JSON format is preserved; `.tasks.npz` records planner data that it does not contain.

MuJoCo actuators accept **joint position targets**. A ZMP is not a motor torque and
cannot be copied into `data.ctrl`. Do not overwrite `qpos` after initialization.

In [ ]:
def joint_mapping(reference_names,robot_names):
    """Return reference indices in robot actuator order; allow only the two antenna extras."""
    # TODO
    raise NotImplementedError("joint_mapping")

In [ ]:
# TEST CELL (A)
robot=Duck();ref_names=['right_antenna']+list(reversed(robot.names))+['left_antenna']
indices=joint_mapping(ref_names,robot.names)
assert [ref_names[i] for i in indices]==robot.names
for bad in [ref_names[:-2],ref_names+['mystery'],ref_names+['left_hip_yaw']]:
    try:joint_mapping(bad,robot.names)
    except ValueError:pass
    else:raise AssertionError('Invalid mapping accepted')
reference=generate_reference()
assert tuple(reference.contacts[0])==(1,1) and tuple(reference.contacts[-1])==(1,1)
assert np.all(np.isfinite(reference.com))
assert reference.time[-1]>7
print('Part A tests passed; finite task reference generated')

## Part B: LIPM MPC with future supports (30 points)

For $x=[c_x,c_y,\dot c_x,\dot c_y]^T$, $p=[p_x,p_y]^T$, and fixed height $h$,
$$\ddot{\mathbf c}=\omega^2(\mathbf c-\mathbf p),\quad \omega=\sqrt{g/h}.$$
With $C=\cosh(\omega\Delta t)$ and $S=\sinh(\omega\Delta t)$,
$$A=\begin{bmatrix}CI&(S/\omega)I\\\omega SI&CI\end{bmatrix},\quad
B=\begin{bmatrix}(1-C)I\\-\omega SI\end{bmatrix}.$$

This time eliminate the equality-constrained predicted states to improve numerical
conditioning: $X=Fx_0+LU$, with $U=[p_0^T,\ldots,p_{N-1}^T]^T$.
$F$ begins with $I$ and $L$ begins with a zero block; advance each block using $A$ and insert $B$.
This is algebraically the same dynamics constraint used in Lecture 7's larger sparse QP.

Minimize
$$\|Fx_0+LU-X_{ref}\|_{\bar Q}^2+
\|U-U_{ref}\|_{\bar R}^2+\|DU-d_{prev}\|_{\bar W}^2.$$
Use $Q=\operatorname{diag}(20,20,1,1)$ for all $N+1$ states, $R=0.1I$, and
$\bar W=0.02I$. $D$ takes first differences; $d_{prev}=[p_{previous}^T,0,\ldots,0]^T$.

$$H=2(L^T\bar QL+\bar R+D^T\bar WD),$$
$$f=2L^T\bar Q(Fx_0-X_{ref})-2\bar R U_{ref}-2D^T\bar Wd_{prev}.$$
Divide **both** $H,f$ by the largest diagonal entry of $H$; positive scalar scaling
preserves the optimizer. Keep the discarded cost constant out of the QP.
For each time step, turn its support polygon into halfspaces $A_ip_i\le b_i$ using
`ConvexHull(poly).equations`. Stack these block-diagonally. ZMP support limits stay hard.
There are no artificial state limits that would make this diagnostic QP infeasible simply
because the full-body state differs from the simplified model.

`walking_qp` returns `(H, f, G, lower, upper)` in the OSQP format.
Measured CoM velocity is low-pass filtered. References and support polygons are translated
into the measured stance frame at each planned support transition; this same alignment is
used for both baseline and MPC. Global travel is measured separately, not inferred from
aligned tracking error.

In [ ]:
def lipm_matrices(height,dt):
    """Return exact discrete A (4x4), B (4x2)."""
    # TODO
    raise NotImplementedError("lipm_matrices")

In [ ]:
def prediction_matrices(A,B,N):
    """Return F and L for X=F x0+L U, including the initial state."""
    # TODO
    raise NotImplementedError("prediction_matrices")

In [ ]:
def walking_qp(A,B,state,reference,zmp_ref,polygons,previous):
    """Assemble the scaled condensed QP with hard future support constraints."""
    # TODO
    raise NotImplementedError("walking_qp")

In [ ]:
# TEST CELL (B): independent rollout and objective checks.
from scipy.linalg import expm
A,B=lipm_matrices(.22,.02)
Ac=np.block([[np.zeros((2,2)),np.eye(2)],[(9.81/.22)*np.eye(2),np.zeros((2,2))]])
Bc=np.vstack([np.zeros((2,2)),-(9.81/.22)*np.eye(2)])
E=expm(np.block([[Ac,Bc],[np.zeros((2,6))]])*.02)
np.testing.assert_allclose(A,E[:4,:4],atol=1e-12);np.testing.assert_allclose(B,E[:4,4:],atol=1e-12)
N=5;F,L=prediction_matrices(A,B,N);state=np.array([.01,-.01,.02,0]);rng=np.random.default_rng(12)
U=rng.normal(scale=.01,size=(N,2));X=[state]
for u in U:X.append(A@X[-1]+B@u)
np.testing.assert_allclose((F@state+L@U.ravel()).reshape(N+1,4),X,atol=1e-12)
poly=np.array([[-.03,-.02],[.03,-.02],[.03,.02],[-.03,.02]])
refs=np.zeros((N+1,4));prefs=np.zeros((N,2));previous=np.array([.005,0])
H,f,G,lo,hi=walking_qp(A,B,state,refs,prefs,[poly]*N,previous)
def direct_cost(flat):
    inputs=flat.reshape(N,2);x=state.copy();cost=x@np.diag([20,20,1,1])@x;old=previous
    for u in inputs:
        cost+=.1*(u@u)+.02*np.sum((u-old)**2);old=u;x=A@x+B@u
        cost+=x@np.diag([20,20,1,1])@x
    return cost
basis=np.eye(2*N)[0];zero=np.zeros(2*N)
scale=(direct_cost(basis)+direct_cost(-basis)-2*direct_cost(zero))/H[0,0]
for z in [U.ravel(),rng.normal(scale=.02,size=2*N)]:
    np.testing.assert_allclose(.5*z@(H@z)+f@z,(direct_cost(z)-direct_cost(zero))/scale,atol=1e-10)
result=solve_qp(H,f,G,lo,hi)
assert np.all(G@result.z<=hi+2e-5)
print('Part B tests passed')

## Part C: Close the loop in MuJoCo (25 points)

Complete `walking_action`: assemble the QP, call `solve_qp` with `eps=1e-6` and
`max_iter=100000`, return the first ZMP, all predicted states, and the QPResult.
The runner only applies the first action and re-solves from the next measured state.

The provided inner adapter is **bounded kinematic compliance**, not a torque optimizer:
$$\delta c_{target}=-0.15(p_{MPC}-p_{ref}),\qquad |\delta c_{target}|\le0.004\ \mathrm m.$$
A first-order filter uses $\delta c_k=0.8\delta c_{k-1}+0.2\delta c_{target}$.
IK tracks the nominal CoM plus this offset and the scheduled feet, starting from measured
joint state. Only joint targets are actuated; the virtual IK root is never imposed on the
physical robot. The sign/gain of this adapter is empirically tuned for this model; it does
not imply exact ZMP realization or whole-body optimality. Explain this limitation in your report.

The runner terminates on a QP/IK failure, non-foot ground contact, trunk tilt above
30 degrees, or base height below 65% of its initial height. No failure is silently replaced
with a canned motion. The position servo retains joint and force limits and normal friction.

In [ ]:
def walking_action(state,reference,zmp_ref,polygons,previous,height=.22,dt=.02):
    """Return first ZMP, predicted state sequence, QPResult."""
    # TODO
    raise NotImplementedError("walking_action")

In [ ]:
# Given experiment calls: identical reference, model and initial state.
baseline=run_experiment(reference,None,'baseline',record=True)
mpc_run=run_experiment(reference,walking_action,'mpc',record=True)
for result in (baseline,mpc_run):
    save_result(result,OUTPUT)
    print(json.dumps({k:v for k,v in result['summary'].items() if k!='landings'},indent=2))
plot_results([baseline,mpc_run]);plt.show()

In [ ]:
# TEST CELL (C): physics acceptance, not playback.
s=mpc_run['summary']
assert s['failure'] is None,s['failure']
assert s['duration_s']>=7
assert s['alternating_landings_after_start']>=8
assert s['forward_distance_after_start_m']>=.08
assert s['max_tilt_deg']<=30 and s['min_base_height_ratio']>=.65
assert s['max_qp_residual']<2e-5 and s['max_ik_error_m']<.01
assert s['max_joint_violation_rad']<.002
assert s['success']
print('Part C: physical walking acceptance passed')

In [ ]:
# Given export: existing course JSON plus task-space sidecar, never implicit wrapping.
episode=export_reference(reference,Duck(),OUTPUT/'open_duck_midterm.json')
indices=joint_mapping(episode['Joints'],robot.names)
frame=interpolate_episode(episode,.015)
assert len(frame['joints'][indices])==14
assert np.isclose(np.linalg.norm(frame['root_quat_wxyz']),1)
if os.environ.get('COURSE_RENDER_VIDEO','0')=='1':
    render_video(mpc_run,OUTPUT/'mpc_walk.mp4')
print('Saved course JSON, task sidecar and measured simulation logs to',OUTPUT)

## Part D: Explain the evidence (20 points)

1. Compare duration, forward travel, measured alternating landings, CoM/joint RMSE,
   trunk tilt, actuator saturation, contact mismatch and solver time on the same interval.
2. Explain how the initial-state measurement and future support sequence enter the QP.
3. What changes when the horizon is shortened? Change only N in a separate experiment,
   preserving the formal test settings. Report timing and all failures honestly.
4. Distinguish planned ZMP, ZMP reconstructed from LIPM, and a CoP inferred from physical
   contact forces. Which did this controller constrain? Why is that insufficient for full feasibility?
5. Explain the 16-to-14 joint mapping, coordinate conventions, and why directly assigning
   the root pose would invalidate a physics walking claim.
6. Discuss stance-frame alignment and bounded inner-loop corrections. Is the resulting
   controller globally optimal for the full robot? Does this experiment establish push recovery?

**Write your analysis here.**

## Submission (5 points)
Submit this completed notebook, PDF report, the generated course JSON and `.tasks.npz`,
metrics JSON, simulation logs, and a short physical-simulation video. Include exact package
versions and changed parameters. Keep outputs out of Git and submit them separately.

Run from a clean kernel. Do not submit `solution/`. Basic success means at least six seconds
after the one-second startup allowance, eight measured alternating forward landings, and
0.08 m forward travel after startup, with the physical and solver checks above.
A landing requires positive normal force, at least 30 ms airborne, foot-site height at least
7 mm while airborne, 20 ms renewed contact, and at least 4 mm forward foot displacement.
These thresholds reject contact chatter; contacts are never inferred from the planned flags.

Optional extension: apply a documented small external push and compare recovery with the
same push in the baseline. Do not modify the required nominal acceptance test.